# Week 11: GIS Integration of PINN Safety Penalty Layer

> **Milestone:** Integrate PINN safety penalty as spatially-varying GIS raster layer
> **Deliverable:** Notebook 08 complete - full-resolution safety penalty map + integrated decision grid

*From PROJECT_BRIEF.md - Phase 3: PINN Integration (Week 11)*

**Goal:** Extend the PINN safety penalty from 5 candidate points (Week 10 tactical analysis) to a full-resolution raster across the entire Southern Africa ROI, then overlay it on the strategic screening heatmap to produce a physics-integrated decision map.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import json
from shapely.geometry import box
import geopandas as gpd

wd = '/home/recursivex/my_projects/my_career/2026_roadmap_fully_funded_opportunities/hydrogen_storage_site_selection'
os.chdir(wd)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load Strategic Screening Results (Week 8)

Load the normalized data layers, equal-weight strategic scores, and robustness grid from Phase 2.

In [ ]:
# Load normalized data layers
solar_norm = np.load('outputs/solar_norm.npy')
water_norm = np.load('outputs/water_norm.npy')
pop_norm = np.load('outputs/pop_norm.npy')
robust_grid = np.load('outputs/robust_grid.npy')
equal_scores_grid = np.load('outputs/equal_scores_grid.npy')

# Create ROI polygon (Southern Africa: Namibia, Botswana, western South Africa)
roi = box(12.0, -20.0, 22.0, -10.0)
roi_gdf = gpd.GeoDataFrame({'geometry': [roi]}, crs='EPSG:4326')

# Valid pixel mask (overlapping non-zero values across all layers)
mask = (solar_norm > 0) & (water_norm > 0) & (pop_norm > 0)

print(f'ROI grid shape: {solar_norm.shape}')
print(f'Valid pixels: {int(mask.sum())} / {mask.size}')
print(f'Strategic score range: {equal_scores_grid[mask].min():.4f} - {equal_scores_grid[mask].max():.4f}')

## Load PINN Model (Week 9-10, retrained v3)

Load the retrained physics-informed PINN (v3) trained on real experimental H2-brine IFT data.
Inputs are normalized using training-set statistics; outputs are denormalized to mN/m.

In [ ]:
# Load normalization statistics from v3 training
with open('outputs/pinn_norm_stats_v3.json', 'r') as f:
    norm_stats = json.load(f)

P_mean, P_std = norm_stats['P_mean'], norm_stats['P_std']
T_mean, T_std = norm_stats['T_mean'], norm_stats['T_std']
S_mean, S_std = norm_stats['S_mean'], norm_stats['S_std']
IFT_mean, IFT_std = norm_stats['IFT_mean'], norm_stats['IFT_std']

# PINN Architecture (matches v3 training)
class PINNIFTS(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64, output_dim=1, n_hidden_layers=4):
        super(PINNIFTS, self).__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(n_hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)
        # Learnable PR-EOS parameters
        self.log_Tc = nn.Parameter(torch.log(torch.tensor(33.2)))
        self.log_Pc = nn.Parameter(torch.log(torch.tensor(1.30)))
        self.omega = nn.Parameter(torch.tensor(-0.216))
    
    @property
    def Tc(self): return torch.exp(self.log_Tc)
    @property
    def Pc(self): return torch.exp(self.log_Pc)
    
    def forward(self, x):
        return self.net(x)

# Load trained PINN model v3
model = PINNIFTS()
checkpoint = torch.load('outputs/pinn_ift_model_v3.pt', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f'Loaded PINN v3 trained on experimental H2-brine IFT data')
print(f'Train RMSE: {checkpoint.get("train_rmse", "N/A")} mN/m')
print(f'Val RMSE:   {checkpoint.get("val_rmse", "N/A")} mN/m')
print(f'Learned Tc: {checkpoint.get("learned_Tc", "N/A")} K (lit: 33.2 K)')
print(f'Learned Pc: {checkpoint.get("learned_Pc", "N/A")} MPa (lit: 1.30 MPa)')
print(f'Learned omega: {checkpoint.get("learned_omega", "N/A")} (lit: -0.216)')

def pinn_predict(P_vals, T_vals, S_vals, batch_size=1000):
    """Predict IFT for arrays of P, T, S values."""
    X = np.stack([P_vals, T_vals, S_vals], axis=1).astype(np.float32)
    X_n = ((X - np.array([P_mean, T_mean, S_mean], dtype=np.float32))
           / np.array([P_std, T_std, S_std], dtype=np.float32)).astype(np.float32)
    n_batches = int(np.ceil(len(X) / batch_size))
    preds_n = np.zeros(len(X), dtype=np.float32)
    with torch.no_grad():
        for b in range(n_batches):
            s = b * batch_size
            e = min((b + 1) * batch_size, len(X))
            preds_n[s:e] = model(torch.tensor(X_n[s:e]).to(device)).cpu().numpy().flatten()
    return preds_n * IFT_std + IFT_mean

## Load Reservoir Condition Grids

Reservoir condition grids (Pressure, Temperature, Salinity, Depth) were generated in Week 11 preprocessing by interpolating basin depth and geothermal gradient across the ROI.

In [ ]:
# Load spatially-distributed reservoir conditions
reservoir_P_grid = np.load('outputs/reservoir_P_grid.npy')   # Pressure [MPa]
reservoir_T_grid = np.load('outputs/reservoir_T_grid.npy')   # Temperature [K]
reservoir_S_grid = np.load('outputs/reservoir_S_grid.npy')   # Salinity [ppm]
reservoir_depth_grid = np.load('outputs/reservoir_depth_grid.npy')  # Depth [km]

print(f'Reservoir condition grids loaded:')
print(f'  Pressure [MPa]:   {reservoir_P_grid[mask].min():.2f} - {reservoir_P_grid[mask].max():.2f} (mean {reservoir_P_grid[mask].mean():.2f})')
print(f'  Temperature [K]:  {reservoir_T_grid[mask].min():.1f} - {reservoir_T_grid[mask].max():.1f} (mean {reservoir_T_grid[mask].mean():.1f})')
print(f'  Salinity [ppm]:   {reservoir_S_grid[mask].min():.0f} - {reservoir_S_grid[mask].max():.0f} (mean {reservoir_S_grid[mask].mean():.0f})')
print(f'  Depth [km]:       {reservoir_depth_grid[mask].min():.2f} - {reservoir_depth_grid[mask].max():.2f} (mean {reservoir_depth_grid[mask].mean():.2f})')

## Predict IFT Across Full ROI Grid

Run the PINN at every valid pixel in the ROI to produce a spatially-distributed IFT map. Predictions are run in batches for memory efficiency.

In [ ]:
# Extract valid pixel reservoir conditions
P_vals = reservoir_P_grid[mask]
T_vals = reservoir_T_grid[mask]
S_vals = reservoir_S_grid[mask]

# Run PINN inference across full ROI (normalized inputs, denormalized outputs)
print(f'Running PINN v3 inference on {len(P_vals)} valid pixels...')
ift_preds = pinn_predict(P_vals, T_vals, S_vals)

# Reshape back to full grid
ift_grid = np.zeros_like(solar_norm, dtype=np.float32)
ift_grid[mask] = ift_preds

print(f'IFT prediction complete')
print(f'IFT range across ROI: {ift_preds.min():.2f} - {ift_preds.max():.2f} mN/m (mean {ift_preds.mean():.2f})')

## Create Safety Penalty Layer

High IFT → higher capillary trapping risk → lower storage safety.

The safety penalty normalizes IFT across the ROI to [0, 1], where 1 = highest penalty (worst safety). The safety score is the inverse (1 - penalty).

In [ ]:
# Compute safety penalty from spatially-varying IFT
ift_valid = ift_preds  # valid pixel IFT values
ift_min, ift_max = ift_valid.min(), ift_valid.max()

print(f'IFT range: {ift_min:.2f} - {ift_max:.2f} mN/m')

# Normalize IFT to safety penalty [0, 1]
# Handle degenerate case where model outputs constant IFT (neutral penalty = 0.5)
if ift_max > ift_min:
    safety_penalty_flat = (ift_valid - ift_min) / (ift_max - ift_min)
else:
    safety_penalty_flat = np.full_like(ift_valid, 0.5, dtype=np.float32)
    print('NOTE: Model outputs constant IFT across ROI — using neutral safety penalty = 0.5')
safety_score_flat = 1.0 - safety_penalty_flat

# Reshape back to full grids
safety_penalty_grid = np.zeros_like(solar_norm, dtype=np.float32)
safety_score_grid = np.zeros_like(solar_norm, dtype=np.float32)
safety_penalty_grid[mask] = safety_penalty_flat
safety_score_grid[mask] = safety_score_flat

print(f'Safety penalty range: {safety_penalty_flat.min():.4f} - {safety_penalty_flat.max():.4f}')
print(f'Safety score range:   {safety_score_flat.min():.4f} - {safety_score_flat.max():.4f}')

## Integrate Strategic + Tactical Layers

Combine the strategic screening score (equal-weight suitability) with the PINN-based safety score:

Integrated Score = w_strategic × Strategic_Score + w_safety × Safety_Score

Weights: 0.7 strategic (regional screening) + 0.3 safety (physics-based penalty).

In [ ]:
# Integration weights
w_strategic = 0.7  # Regional strategic screening weight
w_safety = 0.3     # Physics-based safety weight

# Flatten valid pixel values
strategic_flat = equal_scores_grid[mask]
safety_flat = safety_score_grid[mask]

# Compute integrated score
integrated_flat = w_strategic * strategic_flat + w_safety * safety_flat

# Reshape back to full grid
integrated_grid = np.zeros_like(solar_norm, dtype=np.float32)
integrated_grid[mask] = integrated_flat

print(f'Integrated score range: {integrated_flat.min():.4f} - {integrated_flat.max():.4f} (mean {integrated_flat.mean():.4f})')

## Generate Integrated Decision Grid

Apply go/no-go decision thresholds to the integrated score:
- GO: Integrated score >= 0.65 AND safety score >= 0.5
- CONDITIONAL GO: Integrated score >= 0.55
- CONDITIONAL NO-GO: 0.35 <= score < 0.55
- NO-GO: score < 0.35

In [ ]:
# Decision thresholds
GO_THRESH = 0.65
COND_GO_THRESH = 0.55
COND_NOGO_THRESH = 0.35
SAFETY_MIN = 0.5

# Decision codes: 0=NO-GO, 1=Conditional NO-GO, 2=Conditional GO, 3=GO
decision_flat = np.zeros(len(strategic_flat), dtype=np.float32)

go_mask = (integrated_flat >= GO_THRESH) & (safety_flat >= SAFETY_MIN)
cond_go_mask = (integrated_flat >= COND_GO_THRESH) & (~go_mask)
cond_nogo_mask = (integrated_flat >= COND_NOGO_THRESH) & (~go_mask) & (~cond_go_mask)

decision_flat[go_mask] = 3.0
decision_flat[cond_go_mask] = 2.0
decision_flat[cond_nogo_mask] = 1.0
decision_flat[~go_mask & ~cond_go_mask & ~cond_nogo_mask] = 0.0

# Reshape to grid
decision_grid = np.zeros_like(solar_norm, dtype=np.float32)
decision_grid[mask] = decision_flat

# Decision summary
names = {0: 'NO-GO', 1: 'Conditional NO-GO', 2: 'Conditional GO', 3: 'GO'}
for code, name in names.items():
    count = int(np.sum(decision_flat == code))
    pct = count / len(decision_flat) * 100
    print(f'{name}: {count} pixels ({pct:.1f}%)')

print(f'\nValid pixels: {len(decision_flat)}')

## Save All Week 11 Outputs

Export the IFT grid, safety penalty/score grids, integrated grid, and decision grid for use in the paper and Week 12.

In [ ]:
# Save all grids
np.save('outputs/ift_grid.npy', ift_grid)
np.save('outputs/safety_penalty_grid.npy', safety_penalty_grid)
np.save('outputs/safety_score_grid.npy', safety_score_grid)
np.save('outputs/integrated_grid.npy', integrated_grid)
np.save('outputs/decision_grid.npy', decision_grid)

# Save integrated decision metadata
integration_metadata = {
    'roi_bounds': [12.0, -20.0, 22.0, -10.0],
    'grid_shape': list(solar_norm.shape),
    'valid_pixels': int(mask.sum()),
    'weights': {'strategic': w_strategic, 'safety': w_safety},
    'thresholds': {
        'go': GO_THRESH,
        'conditional_go': COND_GO_THRESH,
        'conditional_no_go': COND_NOGO_THRESH,
        'safety_min': SAFETY_MIN
    },
    'ift_range_mN_m': [float(ift_preds.min()), float(ift_preds.max())],
    'integrated_range': [float(integrated_flat.min()), float(integrated_flat.max())],
    'decision_counts': {names[c]: int(np.sum(decision_flat == c)) for c in names},
    'model': 'PINN v3 trained on experimental H2-brine IFT data (outputs/pinn_ift_model_v3.pt)'
}

with open('outputs/week11_integration_metadata.json', 'w') as f:
    json.dump(integration_metadata, f, indent=2)

print('All Week 11 grids and metadata saved to outputs/')

## Visualization: Full-Resolution Integrated Decision Map

Generate a multi-panel figure showing: strategic score, safety penalty, integrated score, and the final decision map.

In [ ]:
# Multi-panel visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

cmap = plt.cm.viridis
decision_cmap = plt.cm.RdYlGn

# Panel 1: Strategic score (equal-weight)
im1 = axes[0,0].imshow(equal_scores_grid, cmap=cmap, vmin=0, vmax=1)
axes[0,0].set_title('Strategic Suitability Score (Equal Weights)', fontsize=12)
axes[0,0].axis('off')
plt.colorbar(im1, ax=axes[0,0], fraction=0.046, pad=0.04)

# Panel 2: Safety penalty (PINN IFT-based)
im2 = axes[0,1].imshow(safety_penalty_grid, cmap='YlOrRd', vmin=0, vmax=1)
axes[0,1].set_title('Safety Penalty Layer (PINN IFT-based)', fontsize=12)
axes[0,1].axis('off')
plt.colorbar(im2, ax=axes[0,1], fraction=0.046, pad=0.04)

# Panel 3: Integrated score
im3 = axes[1,0].imshow(integrated_grid, cmap=cmap, vmin=0, vmax=1)
axes[1,0].set_title('Integrated Score (0.7×Strategic + 0.3×Safety)', fontsize=12)
axes[1,0].axis('off')
plt.colorbar(im3, ax=axes[1,0], fraction=0.046, pad=0.04)

# Panel 4: Decision map
im4 = axes[1,1].imshow(decision_grid, cmap=decision_cmap, vmin=0, vmax=3)
axes[1,1].set_title('Final Go/No-Go Decision Map', fontsize=12)
axes[1,1].axis('off')
cbar = plt.colorbar(im4, ax=axes[1,1], fraction=0.046, pad=0.04, ticks=[0,1,2,3])
cbar.set_ticklabels(['NO-GO', 'Cond. NO-GO', 'Cond. GO', 'GO'])

plt.suptitle('Week 11: PINN Safety Penalty GIS Integration - Southern Africa ROI (v3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/week11_full_integration.png', dpi=150, bbox_inches='tight')
plt.close()

print('Visualization saved to outputs/week11_full_integration.png')

## Week 11 Deliverables Summary

- **Spatially-varying IFT raster:** outputs/ift_grid.npy (full-resolution PINN v3 predictions across ROI)
- **Safety penalty raster:** outputs/safety_penalty_grid.npy (normalized IFT → penalty)
- **Safety score raster:** outputs/safety_score_grid.npy (inverse of penalty)
- **Integrated score raster:** outputs/integrated_grid.npy (0.7×Strategic + 0.3×Safety)
- **Decision raster:** outputs/decision_grid.npy (4-class go/no-go)
- **Metadata:** outputs/week11_integration_metadata.json
- **Visualization:** outputs/week11_full_integration.png

The two-layer framework is now fully operational as GIS rasters across the entire ROI, ready for paper figure generation in Week 12.

**Note:** PINN v3 was retrained on 91 experimental H2-brine IFT data points (Hosseini et al. 2022; Ai & Trusler 2026)
with PR-EOS physics loss. The model now responds to pressure, temperature, and salinity inputs,
producing a spatially-varying safety penalty rather than a neutral constant.